# Binance USD-M Futures Classification Maps

This notebook shows how to get Binance official futures classification dictionaries from `exchangeInfo`:

- `underlyingType`: primary category, such as `COIN`, `EQUITY`, `PREMARKET`, `COMMODITY`, `INDEX`.
- `underlyingSubType`: official sector tags, such as `AI`, `Layer-1`, `Alpha`, `Pre-IPO`, `TradFi`.

The helper returns dictionaries where each key is a category/tag and each value is the sorted list of matching futures symbols.

## 1. Import Local Package

In [2]:
import sys
from pathlib import Path

import pandas as pd


def find_repo_path():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/home/suncong/binance_klines_data_fetch"),
    ]
    for candidate in candidates:
        if (candidate / "binance_klines_data_fetch").is_dir():
            return candidate.resolve()
    raise RuntimeError("Could not find the binance_klines_data_fetch repo path")


repo_path = find_repo_path()
repo_path_str = str(repo_path)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)

from binance_klines_data_fetch import get_um_futures_classification_maps

print("Imported package from:", repo_path)

Imported package from: /home/suncong/binance_klines_data_fetch


## 2. Fetch Classification Maps

This cell calls Binance USD-M Futures `/fapi/v1/exchangeInfo`. By default, it includes `TRADING` contracts with `contractType` equal to `PERPETUAL` or `TRADIFI_PERPETUAL`, so it covers crypto perpetuals plus TradFi and Pre-IPO perpetuals.

In [3]:
classification_maps = get_um_futures_classification_maps(
    quote_assets=None,
    contract_types=("PERPETUAL", "TRADIFI_PERPETUAL"),
    unknown_label="UNKNOWN",
)

type_map = classification_maps["underlyingType"]
subtype_map = classification_maps["underlyingSubType"]

print("underlyingType groups:", len(type_map))
print("underlyingSubType groups:", len(subtype_map))

underlyingType groups: 5
underlyingSubType groups: 22


## 3. underlyingType: Primary Category To Symbols

In [4]:
type_summary = pd.DataFrame(
    [
        {
            "underlyingType": label,
            "symbol_count": len(symbols),
            "sample_symbols": symbols[:20],
        }
        for label, symbols in type_map.items()
    ]
).sort_values(["symbol_count", "underlyingType"], ascending=[False, True])

display(type_summary.reset_index(drop=True))

,underlyingType,symbol_count,sample_symbols
0,COIN,566,"[0GUSDT, 1000000BOBUSDT, 1000000MOGUSDT, 1000B..."
1,EQUITY,48,"[AAPLUSDT, AMDUSDT, AMZNUSDT, ARMUSDT, AVGOUSD..."
2,COMMODITY,8,"[BZUSDT, CLUSDT, COPPERUSDT, NATGASUSDT, XAGUS..."
3,PREMARKET,3,"[OPENAIUSDT, QNTXUSDT, SPCXUSDT]"
4,INDEX,2,"[ALLUSDT, BTCDOMUSDT]"


In [5]:
def symbols_frame(group_map, label):
    return pd.DataFrame({"symbol": group_map.get(label, [])})


for label in ["COIN", "EQUITY", "PREMARKET", "COMMODITY", "INDEX", "UNKNOWN"]:
    symbols = type_map.get(label, [])
    print(f"{label}: {len(symbols)} symbols")
    display(symbols_frame(type_map, label).head(50))

COIN: 566 symbols


,symbol
0,0GUSDT
1,1000000BOBUSDT
2,1000000MOGUSDT
3,1000BONKUSDC
4,1000BONKUSDT
5,1000CATUSDT
6,1000CHEEMSUSDT
7,1000FLOKIUSDT
8,1000LUNCUSDT
9,1000PEPEUSDC


EQUITY: 48 symbols


,symbol
0,AAPLUSDT
1,AMDUSDT
2,AMZNUSDT
3,ARMUSDT
4,AVGOUSDT
5,BABAUSDT
6,BEUSDT
7,BRKBUSDT
8,CBRSUSDT
9,COHRUSDT


PREMARKET: 3 symbols


...

,symbol
0,BZUSDT
1,CLUSDT
2,COPPERUSDT
3,NATGASUSDT
4,XAGUSDT
5,XAUUSDT
6,XPDUSDT
7,XPTUSDT


INDEX: 2 symbols


,symbol
0,ALLUSDT
1,BTCDOMUSDT


UNKNOWN: 0 symbols


,symbol


## 4. underlyingSubType: Official Sector Tag To Symbols

In [6]:
subtype_summary = pd.DataFrame(
    [
        {
            "underlyingSubType": label,
            "symbol_count": len(symbols),
            "sample_symbols": symbols[:20],
        }
        for label, symbols in subtype_map.items()
    ]
).sort_values(["symbol_count", "underlyingSubType"], ascending=[False, True])

display(subtype_summary.reset_index(drop=True))

,underlyingSubType,symbol_count,sample_symbols
0,DeFi,117,"[1000LUNCUSDT, 1INCHUSDT, AAVEUSDT, ACXUSDT, A..."
1,Alpha,71,"[4USDT, ACUUSDT, AIAUSDT, AIGENSYNUSDT, AIOUSD..."
2,Infrastructure,59,"[2ZUSDT, AERGOUSDT, ALTUSDT, API3USDT, ARKMUSD..."
3,TradFi,59,"[AAPLUSDT, AMDUSDT, AMZNUSDT, ARMUSDT, AVGOUSD..."
4,AI,58,"[ACTUSDT, AGTUSDT, AINUSDT, AIOTUSDT, AIXBTUSD..."
5,Layer-1,53,"[0GUSDT, ADAUSDT, ALGOUSDT, APTUSDT, ARKUSDT, ..."
6,Meme,47,"[1000000BOBUSDT, 1000000MOGUSDT, 1000BONKUSDT,..."
7,USDC,37,"[1000BONKUSDC, 1000PEPEUSDC, 1000SHIBUSDC, AAV..."
8,Gaming,26,"[ACEUSDT, AGLDUSDT, BEAMXUSDT, BIGTIMEUSDT, CA..."
9,Layer-2,21,"[ARBUSDT, AUSDT, AZTECUSDT, BTRUSDT, CELRUSDT,..."


In [7]:
for label in ["AI", "Layer-1", "Alpha", "Pre-IPO", "TradFi", "DeFi", "Meme", "UNKNOWN"]:
    symbols = subtype_map.get(label, [])
    print(f"{label}: {len(symbols)} symbols")
    display(symbols_frame(subtype_map, label).head(50))

AI: 58 symbols


,symbol
0,ACTUSDT
1,AGTUSDT
2,AINUSDT
3,AIOTUSDT
4,AIXBTUSDT
5,AKTUSDT
6,ALCHUSDT
7,ALLOUSDT
8,ARCUSDT
9,ATHUSDT


Layer-1: 53 symbols


,symbol
0,0GUSDT
1,ADAUSDT
2,ALGOUSDT
3,APTUSDT
4,ARKUSDT
5,ASTRUSDT
6,ATOMUSDT
7,AVAXUSDT
8,BBUSDT
9,BERAUSDT


Alpha: 71 symbols


...

,symbol
0,1000LUNCUSDT
1,1INCHUSDT
2,AAVEUSDT
3,ACXUSDT
4,AEROUSDT
5,AEVOUSDT
6,ANKRUSDT
7,ASTERUSDT
8,AUCTIONUSDT
9,AVNTUSDT


Meme: 47 symbols


,symbol
0,1000000BOBUSDT
1,1000000MOGUSDT
2,1000BONKUSDT
3,1000CATUSDT
4,1000CHEEMSUSDT
5,1000FLOKIUSDT
6,1000PEPEUSDT
7,1000RATSUSDT
8,1000SATSUSDT
9,1000SHIBUSDT


UNKNOWN: 16 symbols


,symbol
0,ALPINEUSDT
1,ASRUSDT
2,AVAUSDT
3,BIOUSDT
4,BTCUSD1
5,COSUSDT
6,CTRUSDT
7,EULUSDT
8,FFUSDT
9,ICNTUSDT


In [12]:
symbols_frame(subtype_map, 'Alpha')['symbol'].to_list()

['4USDT',
 'ACUUSDT',
 'AIAUSDT',
 'AIGENSYNUSDT',
 'AIOUSDT',
 'AKEUSDT',
 'APRUSDT',
 'ARIAUSDT',
 'ATUSDT',
 'BASEDUSDT',
 'BASUSDT',
 'BEATUSDT',
 'BILLUSDT',
 'BIRBUSDT',
 'BLESSUSDT',
 'BLUAIUSDT',
 'BSBUSDT',
 'BULLAUSDT',
 'CARVUSDT',
 'CLANKERUSDT',
 'CLOUSDT',
 'COAIUSDT',
 'COLLECTUSDT',
 'CYSUSDT',
 'DOODUSDT',
 'ELSAUSDT',
 'ESPUSDT',
 'FIGHTUSDT',
 'FOLKSUSDT',
 'GENIUSUSDT',
 'GUAUSDT',
 'GWEIUSDT',
 'HANAUSDT',
 'INUSDT',
 'INXUSDT',
 'IRYSUSDT',
 'JCTUSDT',
 'JELLYJELLYUSDT',
 'KGENUSDT',
 'LABUSDT',
 'LYNUSDT',
 'MAGMAUSDT',
 'NIGHTUSDT',
 'ONUSDT',
 'OPGUSDT',
 'PHAROSUSDT',
 'PIEVERSEUSDT',
 'POWERUSDT',
 'PRLUSDT',
 'PTBUSDT',
 'QUSDT',
 'RAVEUSDT',
 'RIVERUSDT',
 'ROBOUSDT',
 'SAPIENUSDT',
 'SIRENUSDT',
 'SKRUSDT',
 'SPACEUSDT',
 'SPORTFUNUSDT',
 'STABLEUSDT',
 'STARUSDT',
 'TRADOORUSDT',
 'TRIAUSDT',
 'TRUTHUSDT',
 'UAIUSDT',
 'UBUSDT',
 'USUSDT',
 'WETUSDT',
 'XNYUSDT',
 'XPINUSDT',
 'ZKPUSDT']

## 5. Use The Dictionaries Directly

In [7]:
ai_symbols = subtype_map.get("AI", [])
layer1_symbols = subtype_map.get("Layer-1", [])
pre_ipo_symbols = subtype_map.get("Pre-IPO", [])
coin_symbols = type_map.get("COIN", [])

print("AI symbols:", ai_symbols[:30])
print("Layer-1 symbols:", layer1_symbols[:30])
print("Pre-IPO symbols:", pre_ipo_symbols[:30])
print("COIN symbols:", coin_symbols[:30])

selected_symbols = sorted(set(ai_symbols) | set(layer1_symbols) | set(pre_ipo_symbols))
print("Combined selected symbols:", len(selected_symbols))
print(selected_symbols[:50])

AI symbols: ['ACTUSDT', 'AGTUSDT', 'AINUSDT', 'AIOTUSDT', 'AIXBTUSDT', 'AKTUSDT', 'ALCHUSDT', 'ALLOUSDT', 'ARCUSDT', 'ATHUSDT', 'AVAAIUSDT', 'AWEUSDT', 'CGPTUSDT', 'CHRUSDT', 'CLANKERUSDT', 'COOKIEUSDT', 'CUSDT', 'FETUSDT', 'FHEUSDT', 'FLOCKUSDT', 'FLUXUSDT', 'GOATUSDT', 'GRASSUSDT', 'GRIFFAINUSDT', 'GRTUSDT', 'HOLOUSDT', 'IDOLUSDT', 'IOUSDT', 'IPUSDT', 'KITEUSDT']
Layer-1 symbols: ['0GUSDT', 'ADAUSDT', 'ALGOUSDT', 'APTUSDT', 'ARKUSDT', 'ASTRUSDT', 'ATOMUSDT', 'AVAXUSDT', 'BBUSDT', 'BERAUSDT', 'BNBUSDT', 'CCUSDT', 'CELOUSDT', 'DOTUSDT', 'DUSKUSDT', 'EGLDUSDT', 'ETHUSDT', 'FLOWUSDT', 'HBARUSDT', 'HIVEUSDT', 'ICPUSDT', 'ICXUSDT', 'INITUSDT', 'IOSTUSDT', 'KASUSDT', 'KSMUSDT', 'LUNA2USDT', 'MINAUSDT', 'MOVEUSDT', 'MOVRUSDT']
Pre-IPO symbols: ['OPENAIUSDT', 'QNTXUSDT', 'SPCXUSDT']
COIN symbols: ['0GUSDT', '1000000BOBUSDT', '1000000MOGUSDT', '1000BONKUSDC', '1000BONKUSDT', '1000CATUSDT', '1000CHEEMSUSDT', '1000FLOKIUSDT', '1000LUNCUSDT', '1000PEPEUSDC', '1000PEPEUSDT', '1000RATSUSDT', '1000S

## Optional Filters

Use these variants when you need a narrower universe:

```python
# USDT-quoted contracts only
maps = get_um_futures_classification_maps(quote_assets=["USDT"])

# Crypto perpetual contracts only, excluding TRADIFI_PERPETUAL
maps = get_um_futures_classification_maps(contract_types=("PERPETUAL",))
```